# Build APK — Thích Cầm Rìu (Google Colab)

Notebook này tự động:
1. Cài **Java 17**, **Node.js 20**, **Android SDK** (`platform-tools`, `android-34`, `build-tools;34.0.0`).
2. Clone mã nguồn từ GitHub (`spdatdd/vibe-code-throw-axes`).
3. Chạy `npm run build:web`, tạo project Android bằng **Capacitor**, và build `./gradlew assembleDebug`.
4. Tải file `ThichCamRiu-debug.apk` về máy.

In [ ]:
# @title 1. Cài đặt Java 17, Node.js 20 và Android SDK
%%bash
set -e

echo "==> Cài Java 21 & công cụ..."
apt-get update -qq
apt-get install -y -qq openjdk-21-jdk unzip wget curl git > /dev/null

echo "==> Kiểm tra / cài Node.js 20..."
if ! node -v 2>/dev/null | grep -qE '^v(20|22|24|26)\.'; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null
  apt-get install -y -qq nodejs > /dev/null
fi
node -v
java -version

echo "==> Cài Android Command Line Tools..."
export ANDROID_HOME=/opt/android-sdk
if [ ! -d "$ANDROID_HOME/cmdline-tools/latest" ]; then
  mkdir -p "$ANDROID_HOME/cmdline-tools"
  wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/cmdline-tools.zip
  unzip -q /tmp/cmdline-tools.zip -d "$ANDROID_HOME/cmdline-tools"
  mv "$ANDROID_HOME/cmdline-tools/cmdline-tools" "$ANDROID_HOME/cmdline-tools/latest"
  rm -f /tmp/cmdline-tools.zip
fi

yes | "$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager" --licenses > /dev/null || true
"$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager" "platform-tools" "platforms;android-34" "build-tools;34.0.0"
echo "==> Xong môi trường Android SDK!"

In [ ]:
# @title 2. Clone mã nguồn dự án từ GitHub
%%bash
set -e
REPO_URL="https://github.com/spdatdd/vibe-code-throw-axes.git"
BRANCH="main"

rm -rf /content/throw-axe
git clone --depth 1 -b "$BRANCH" "$REPO_URL" /content/throw-axe
ls -la /content/throw-axe

In [ ]:
# @title 3. Build gói web & đóng gói APK bằng Capacitor + Gradle
%%bash
set -e
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))
if [ -d "/usr/lib/jvm/java-21-openjdk-amd64" ]; then
  export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64
fi
echo "JAVA_HOME=$JAVA_HOME"
export ANDROID_HOME=/opt/android-sdk
export ANDROID_SDK_ROOT=/opt/android-sdk
export PATH="$PATH:$ANDROID_HOME/cmdline-tools/latest/bin:$ANDROID_HOME/platform-tools"

cd /content/throw-axe
npm install --no-audit --no-fund @capacitor/core @capacitor/cli @capacitor/android
npm run build:web
if [ ! -d "android" ]; then
  npx cap add android
else
  npx cap sync android
fi

cd android
chmod +x gradlew
./gradlew assembleDebug

cp app/build/outputs/apk/debug/app-debug.apk /content/ThichCamRiu-debug.apk
ls -lh /content/ThichCamRiu-debug.apk

In [ ]:
# @title 4. Tải file APK về máy
from google.colab import files
files.download('/content/ThichCamRiu-debug.apk')